# Text File Consolidation Tool

This notebook collects all `.txt` files from a specified folder (including subfolders) and consolidates them into a single text file.

**Use Case**: Useful for aggregating multiple text documents from your Sinhala Buddhist corpus into a single file for processing.

## Step 1: Import Required Libraries

In [1]:
import os
from pathlib import Path
from typing import List, Tuple
import chardet  # For detecting file encoding

# Install chardet if not already installed
# !pip install chardet

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 2: Define Configuration Parameters

In [3]:
# Configuration
INPUT_FOLDER = "/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/docai_extractions/2_cleaned_text/බාගතකර_ගැනීම/"
OUTPUT_FILE = "sinhala_pali_texts.txt"
INCLUDE_SUBFOLDERS = True
ADD_FILE_SEPARATORS = True
ENCODING = 'utf-8'

## Step 3: Helper Functions

In [4]:
def detect_encoding(file_path: str) -> str:
    """
    Detect the encoding of a file.
    Useful for Sinhala text which might be in different encodings.

    Args:
        file_path: Path to the file

    Returns:
        Detected encoding as string
    """
    with open(file_path, 'rb') as f:
        raw_data = f.read(10000)  # Read first 10KB for detection
        result = chardet.detect(raw_data)
        return result['encoding'] if result['encoding'] else 'utf-8'


def find_txt_files(folder_path: str, include_subfolders: bool = True) -> List[str]:
    """
    Find all .txt files in a folder.

    Args:
        folder_path: Path to the folder to search
        include_subfolders: Whether to search recursively

    Returns:
        List of file paths
    """
    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    if include_subfolders:
        txt_files = list(folder.rglob('*.txt'))  # Recursive search
    else:
        txt_files = list(folder.glob('*.txt'))  # Non-recursive search

    # Sort files for consistent ordering
    txt_files.sort()

    return [str(f) for f in txt_files]


def read_file_safely(file_path: str, encoding: str = 'utf-8') -> Tuple[str, bool]:
    """
    Read a file with automatic encoding detection if the default fails.

    Args:
        file_path: Path to the file
        encoding: Default encoding to try

    Returns:
        Tuple of (file content, success status)
    """
    try:
        with open(file_path, 'r', encoding=encoding) as f:
            return f.read(), True
    except UnicodeDecodeError:
        # Try to detect encoding
        detected_encoding = detect_encoding(file_path)
        print(f"  ⚠️  Encoding issue detected. Trying {detected_encoding}...")
        try:
            with open(file_path, 'r', encoding=detected_encoding) as f:
                return f.read(), True
        except Exception as e:
            print(f"  ❌ Failed to read file: {e}")
            return "", False
    except Exception as e:
        print(f"  ❌ Error reading file: {e}")
        return "", False

## Step 4: Main Consolidation Function

In [5]:
def consolidate_text_files(
    input_folder: str,
    output_file: str,
    include_subfolders: bool = True,
    add_separators: bool = True,
    encoding: str = 'utf-8'
) -> dict:
    """
    Consolidate all .txt files from a folder into a single file.

    Args:
        input_folder: Path to the folder containing .txt files
        output_file: Name of the output consolidated file
        include_subfolders: Whether to search recursively
        add_separators: Whether to add separators between files
        encoding: Default encoding for reading/writing files

    Returns:
        Dictionary with statistics about the consolidation process
    """
    print("=" * 70)
    print("TEXT FILE CONSOLIDATION TOOL")
    print("=" * 70)
    print(f"\n📁 Source Folder: {input_folder}")
    print(f"📄 Output File: {output_file}")
    print(f"🔍 Include Subfolders: {include_subfolders}\n")

    # Find all text files
    print("🔎 Searching for .txt files...")
    txt_files = find_txt_files(input_folder, include_subfolders)

    if not txt_files:
        print("\n⚠️  No .txt files found in the specified folder.")
        return {"files_found": 0, "files_processed": 0, "files_failed": 0}

    print(f"✅ Found {len(txt_files)} .txt file(s)\n")

    # Consolidate files
    files_processed = 0
    files_failed = 0
    total_chars = 0

    with open(output_file, 'w', encoding=encoding) as outfile:
        for idx, file_path in enumerate(txt_files, 1):
            file_name = os.path.basename(file_path)
            print(f"[{idx}/{len(txt_files)}] Processing: {file_name}")

            # Read file content
            content, success = read_file_safely(file_path, encoding)

            if success:
                # Add separator if enabled
                if add_separators:
                    separator = f"\n\n{'='*70}\n"
                    separator += f"FILE: {file_name}\n"
                    separator += f"PATH: {file_path}\n"
                    separator += f"{'='*70}\n\n"
                    outfile.write(separator)

                # Write content
                outfile.write(content)
                outfile.write("\n\n")  # Add spacing between files

                files_processed += 1
                total_chars += len(content)
                print(f"  ✅ Successfully added ({len(content):,} characters)")
            else:
                files_failed += 1

    # Print summary
    print("\n" + "=" * 70)
    print("CONSOLIDATION COMPLETE")
    print("=" * 70)
    print(f"\n📊 Summary:")
    print(f"   • Files found: {len(txt_files)}")
    print(f"   • Files successfully processed: {files_processed}")
    print(f"   • Files failed: {files_failed}")
    print(f"   • Total characters: {total_chars:,}")
    print(f"   • Output file: {output_file}")
    print(f"   • Output file size: {os.path.getsize(output_file) / 1024:.2f} KB\n")

    return {
        "files_found": len(txt_files),
        "files_processed": files_processed,
        "files_failed": files_failed,
        "total_characters": total_chars,
        "output_file": output_file
    }

## Step 5: Run the Consolidation

In [6]:
# Execute the consolidation
stats = consolidate_text_files(
    input_folder=INPUT_FOLDER,
    output_file=OUTPUT_FILE,
    include_subfolders=INCLUDE_SUBFOLDERS,
    add_separators=ADD_FILE_SEPARATORS,
    encoding=ENCODING
)

TEXT FILE CONSOLIDATION TOOL

📁 Source Folder: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/docai_extractions/2_cleaned_text/බාගතකර_ගැනීම/
📄 Output File: sinhala_pali_texts.txt
🔍 Include Subfolders: True

🔎 Searching for .txt files...
✅ Found 918 .txt file(s)

[1/918] Processing: page_001.txt
  ✅ Successfully added (1,342 characters)
[2/918] Processing: page_002.txt
  ✅ Successfully added (1,445 characters)
[3/918] Processing: page_003.txt
  ✅ Successfully added (1,375 characters)
[4/918] Processing: page_004.txt
  ✅ Successfully added (1,301 characters)
[5/918] Processing: page_005.txt
  ✅ Successfully added (1,341 characters)
[6/918] Processing: page_006.txt
  ✅ Successfully added (1,245 characters)
[7/918] Processing: page_007.txt
  ✅ Successfully added (1,258 characters)
[8/918] Processing: page_008.txt
  ✅ Successfully added (1,430 characters)
[9/918] Processing: page_009.txt
  ✅ Successfully added (1,050 characters)
[10/918] Processing: pa

## Step 6: Preview the Consolidated File (Optional)

In [7]:
# Preview first 1000 characters of the consolidated file
print("\n📄 Preview of consolidated file (first 1000 characters):\n")
print("=" * 70)

with open(OUTPUT_FILE, 'r', encoding=ENCODING) as f:
    preview = f.read(1000)
    print(preview)
    if len(preview) == 1000:
        print("\n... (truncated)")


📄 Preview of consolidated file (first 1000 characters):



FILE: page_001.txt
PATH: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/docai_extractions/2_cleaned_text/බාගතකර_ගැනීම/page_001.txt

V

පෙරවදන

"සීහළවත්ථු" නමැති පාළිපොත වනාහි විනාශමුඛයෙන්
යන්තම් බේරා ගන්නා ලද්දකි. මෙහි ලක්දිව පිළිබඳ කථා රාශියක්
ඇතත් සිංහල අකුරෙන් ලියැවුණු පොතක් තවම දකින්ට
නොලැබුණි. කුඩා කාලයේදී සීහළවත්ථු - සහස්සවත්ථු යන
පොත්දෙකේ නම් වත් මට අසන්ට නොලැබුණි. නමුත් 1908
වර්ෂයේ මා රැන්ගුන් නුවර සිටියදී මේ නම් දෙකම මට දකින්ට
ලැබුණි. ඒවා දකින්ට ලැබුණේ ඒ නුවර මහා විද්‍යාලය භූමියෙහි
තිබුණු බර්නාඩ් නිදහස් පුස්තකාලයේ පොත් ලැයිස්තු වලිනි. ඒ
කාලයේදී නිතර මා ඒ පුස්තකාලයට ගොස් දුර්ලභ පොත් කීපයක්
පිටපත් කරගත් නමුත් මාගේ සැලකිල්ල වැඩිපුර යොමුවී තිබුණේ
අභිධර්මය පිළිබඳ පොත් කෙරෙහිය. පරමත්‍ථවිනිච්ඡය හා එහි
ටීකාව, අභිධම්මාවතාරයේ සංක්‍ෂිප්තටීකාව, නාමරූප
පරිච්ඡේදටීකාව ආදි පොත් කීපයක් මම සියතින්ම පිටපත්
කරගතිමි. 

... (truncated)


## Additional Utilities

### Count lines and words in the consolidated file

In [ ]:
def analyze_consolidated_file(file_path: str) -> dict:
    """
    Analyze the consolidated file for basic statistics.
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    lines = content.split('\n')
    words = content.split()

    stats = {
        "total_lines": len(lines),
        "total_words": len(words),
        "total_characters": len(content),
        "file_size_kb": os.path.getsize(file_path) / 1024
    }

    print("\n📊 File Analysis:")
    print(f"   • Total lines: {stats['total_lines']:,}")
    print(f"   • Total words: {stats['total_words']:,}")
    print(f"   • Total characters: {stats['total_characters']:,}")
    print(f"   • File size: {stats['file_size_kb']:.2f} KB")

    return stats

# Run analysis
if os.path.exists(OUTPUT_FILE):
    file_stats = analyze_consolidated_file(OUTPUT_FILE)

---

## Notes

**Encoding Handling**: The script automatically detects file encoding if UTF-8 fails, which is crucial for Sinhala text that may be saved in different encodings.

**File Separators**: Enable `ADD_FILE_SEPARATORS` to maintain traceability of which content came from which source file.

**Use Cases for Your Project**:
- Consolidating multiple Tripiṭaka text files for preprocessing
- Merging Sinhala corpus files before tokenization
- Creating unified datasets for model training

**Next Steps**:
1. Run text cleaning/preprocessing on the consolidated file
2. Perform tokenization analysis
3. Split into train/validation/test sets for model fine-tuning